# CPU-only verification of observed sub-01 inputs

Select **Runtime > Change runtime type > Hardware accelerator: None**, with **High-RAM**. Run this one cell. It uses your existing EEG files, image directories, feature caches and saved `pilot_sub01_download_plan.json`. EEG hashes and all referenced image paths are checked, which can take time over Drive. Existing data are never downloaded, overwritten or uploaded. Only verification reports/latest-report pointer are written to Drive.

This is verification only: no EEG conversion, feature extraction or training. The new code recognizes the documented 300-point metadata axis accompanying 250 EEG samples. It reports both axes and detailed image failure reasons. A successful process exit does not by itself mean images/features are ready: check `ready_images_and_features` in the printed report. If false, share the concise reasons/counts, not data arrays. Stop here after the run.


In [ ]:
import os, subprocess, sys, tempfile
from pathlib import Path
import torch
assert not torch.cuda.is_available(), "Select Hardware accelerator: None before running."
from google.colab import drive
drive.mount('/content/drive')
# Fresh runtime-only checkout prevents stale imported verification code.
REPO = Path(tempfile.mkdtemp(prefix='eeg-cpu-verify-', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'audit/phase1-reproducible-alignment',
                'https://github.com/firozehrazavi/eeg-alignment-phase1.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface-hub==0.26.2',
                'psutil==6.1.0', 'filelock==3.16.1'], check=True)
RAW = Path('/content/drive/MyDrive/THINGS-EEG2')
CACHE = Path('/content/drive/MyDrive/eeg_results/cache')
env = dict(os.environ, CUDA_VISIBLE_DEVICES='')
subprocess.run([sys.executable, '-m', 'phase1.pilot_inputs', '--root', str(RAW),
                '--cache', str(CACHE), '--image-roots', str(RAW),
                '--verify-existing-sub01'], cwd=REPO, env=env, check=True)
print('STOP: CPU verification finished. Review the readiness report; no preparation or GPU work was started.')
